In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.dummy import DummyClassifier
import numpy as np

In [2]:
# Loand csv file
df = pd.read_csv('users_behavior.csv')

In [3]:
print(df.head())
print("================================")
print(df.info())
print("================================")
print(df.describe())

   calls  minutes  messages   mb_used  is_ultra
0   40.0   311.90      83.0  19915.42         0
1   85.0   516.75      56.0  22696.96         0
2   77.0   467.66      86.0  21060.45         0
3  106.0   745.53      81.0   8437.39         1
4   66.0   418.74       1.0  14502.75         0
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3214 entries, 0 to 3213
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   calls     3214 non-null   float64
 1   minutes   3214 non-null   float64
 2   messages  3214 non-null   float64
 3   mb_used   3214 non-null   float64
 4   is_ultra  3214 non-null   int64  
dtypes: float64(4), int64(1)
memory usage: 125.7 KB
None
             calls      minutes     messages       mb_used     is_ultra
count  3214.000000  3214.000000  3214.000000   3214.000000  3214.000000
mean     63.038892   438.208787    38.281269  17207.673836     0.306472
std      33.236368   234.569872    36.148326   7570.968246  

¿Cuál será el target?
El target seria el `is_ultra`

¿Cuál serán las features?
`calls`
`minutes`
`messages`
`mb_used`

In [4]:
# Separar los features del target  x , y
features=df.drop('is_ultra', axis=1)
target=df['is_ultra']

In [13]:

# Separar los datos en entrenamiento y prueba
X_temp, X_test, y_temp, y_test = train_test_split(
    features, target, test_size=0.25, random_state=12345)
# Separar los datos en prueba y validacion 
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.33, random_state=12345)

print("Tamaño del dataset de entrenamiento: ", X_train.shape)
print("Tamaño del dataset de prueba: ", X_test.shape)
print("Tamaño del dataset de validación: ", X_val.shape)

print(y_train.value_counts(normalize=True))

Tamaño del dataset de entrenamiento:  (1614, 4)
Tamaño del dataset de prueba:  (804, 4)
Tamaño del dataset de validación:  (796, 4)
is_ultra
0    0.698265
1    0.301735
Name: proportion, dtype: float64


Probar diferentes modelos, arbol de decision, regresión logistica, bosque aleatorio. Se elegira un hiperparametro para balancear los datos ya que estan desbalanceados, esto se aplicará en todos los modelos a entrenar.

In [6]:
best_score = 0
best_depth=0
for depth in range(1, 11):
    clf = DecisionTreeClassifier(class_weight='balanced',max_depth=depth, random_state=12345)
    clf.fit(X_train, y_train)

    # Predicciones en el conjunto de validación
    valid_predictions = clf.predict(X_val)
    valid_accuracy = accuracy_score(y_val, valid_predictions)
    
    if valid_accuracy > best_score:
        best_score = valid_accuracy
        best_depth = depth
        best_model = clf
        
train_accuracy = best_model.score(X_train, y_train)
# Imprimir los resultados
print(f"""
Mejor max_depth: {best_depth}
Accuracy en entrenamiento: {train_accuracy}
Accuracy en validación: {best_score}
""")


Mejor max_depth: 5
Accuracy en entrenamiento: 0.8073110285006195
Accuracy en validación: 0.7889447236180904



In [7]:
best_score = 0
best_est = 0
for est in range(10, 101, 10):  # selecciona el rango del hiperparámetro
    for depth in range(1,11):
    # configura el número de árboles
        model = RandomForestClassifier(class_weight='balanced',random_state=54321,max_depth=depth, n_estimators=est)
        model.fit(X_train, y_train)
        # Predicciones en el conjunto de validación
        valid_predictions = model.predict(X_val)
        valid_accuracy = accuracy_score(y_val, valid_predictions)
        if valid_accuracy > best_score:
            best_score = valid_accuracy
            best_est = est
            best_depth = depth
            best_model = model

train_accuracy = best_model.score(X_train, y_train)


print(f"""
Mejor n_estimators: {best_est}
Mejor max_depth: {best_depth}
Accuracy en entrenamiento: {train_accuracy}
Accuracy en validación: {best_score}
""")


Mejor n_estimators: 100
Mejor max_depth: 7
Accuracy en entrenamiento: 0.8760842627013631
Accuracy en validación: 0.8203517587939698



In [8]:
best_score  =0
best_c  =0
best_model  =0
c_Val=[0.01, 0.1, 1, 10, 100]
for c in c_Val:
    model = LogisticRegression(
        C=c, random_state=54321, solver='liblinear', class_weight='balanced')
    model.fit(X_train, y_train)

    val_accuracy = model.score(X_val, y_val)

    if val_accuracy > best_score:
        best_score = val_accuracy
        best_c = c
        best_model = model
        best_val_accuracy = val_accuracy

train_accuracy = best_model.score(X_train, y_train)

print(f"Mejor C: {best_c}, accuracy en entrenamiento: {train_accuracy}, accuracy en validación: {best_val_accuracy}")

Mejor C: 1, accuracy en entrenamiento: 0.6133828996282528, accuracy en validación: 0.614321608040201


## Resultados
| Modelo                 | Accuracy Entrenamiento | Accuracy Validación | 
|------------------------|----------------------------------|-------------------------|
| DecisionTreeClassifier | 0.8073                  | 0.7889               | 
| RandomForestClassifier |  0.8073                  | **0.8204**           | 
| LogisticRegression     |  0.6134                  | 0.6143               | 


🔸 **Regresión logística**
- Es el más simple y estable (accuracy muy parecida entre entrenamiento y validación).
- Pero no cumple con el umbral de 0.75 → ❌ descartado para este proyecto.

🔸 **Árbol de decisión**
- Accuracy alta en entrenamiento → buen aprendizaje.
- Pero hay una caída visible en validación → posible sobreajuste.

✅ **Random Forest**
- Accuracy alta en entrenamiento ✅
- Accuracy más alta en validación ✅
- Menor caída entre train/val → generaliza mejor
- Cumple sobradamente el objetivo de 0.75

### Comprobar la calidad del modelo usando el conjunto de prueba.

In [9]:
modelrf = RandomForestClassifier(class_weight='balanced',random_state=54321,max_depth=7, n_estimators=100)
modelrf.fit(X_train, y_train)
test_accuracy = modelrf.score(X_test, y_test)
print(f"Accuracy en prueba: {test_accuracy}")

Accuracy en prueba: 0.7873134328358209


### Realizar pruebas de cordura 

In [10]:
# Barajar los valores reales de y_train (pero mantener X igual)
y_shuffled = np.random.permutation(y_train)

# Entrenar modelo con datos revueltos
model_shuffled = RandomForestClassifier(random_state=54321)
model_shuffled.fit(X_train, y_shuffled)

# Evaluar con X_test, y_test original
accuracy_shuffled = model_shuffled.score(X_test, y_test)
print("Accuracy con etiquetas barajadas:", accuracy_shuffled)

Accuracy con etiquetas barajadas: 0.6716417910447762


In [12]:


# Siempre predice lo más común
dummy = DummyClassifier(strategy='most_frequent')
dummy.fit(X_train, y_train)
print("Accuracy del modelo base (tonto):", dummy.score(X_test, y_test))

Accuracy del modelo base (tonto): 0.7002487562189055


# CONCLUSIONES DEL EJERCICIO

- **Accuracy en prueba: 0.7873134328358209**
- Accuracy en validación: **0.8203**
- Accuracy con etiquetas barajadas (modelo aleatorio): **0.6716**
- Accuracy del modelo base (DummyClassifier): **0.7002**



🔀🌲🌲🌲 RandomForestClassifier sigue siendo el mejor modelo, incluso con class_weight='balanced'.

✅ La diferencia entre validación y prueba es mínima, lo que muestra buena generalización.

✅ El modelo real supera claramente al dummy y al modelo con etiquetas barajadas.

🔍 LogisticRegression no alcanza el umbral mínimo de 0.75 → no es competitivo en este caso